# Introduction

Brick kilns are a significant source of air pollution in India, and monitoring their temporal patterns is crucial for environmental management. Traditional approaches require manual inspection of satellite imagery across multiple years, which is time-consuming and prone to inconsistency.

In this notebook, we demonstrate how **Gemini 3 Pro's multi-image context capability** can analyze temporal changes in a single API call. We have satellite images of the same location (28.212481°N, 77.401398°E) captured across multiple years (2014-2022), and we ask Gemini to:

1. Identify when brick kilns first appear
2. Determine the type of kilns (e.g., Bull's Trench Kiln, Fixed Chimney Kiln, Zig-zag kiln)
3. Track changes in kiln count and configuration over time
4. Detect patterns in kiln operations (active vs. inactive periods)

This approach leverages Gemini's ability to process multiple images in a single context, eliminating the need for sequential processing and maintaining temporal coherence across the analysis.

# Setup

In [ ]:
import os
import glob
import re
from google import genai
from PIL import Image
import matplotlib.pyplot as plt
import json

# Initialize Gemini client
if 'GEMINI_API_KEY' not in os.environ:
    raise ValueError(
        "GEMINI_API_KEY not found in environment.\n"
        "Set it with: export GEMINI_API_KEY='your-key'\n"
        "Get your key at: https://aistudio.google.com/apikey"
    )

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
MODEL = "models/gemini-3-pro-preview"

print(f"✓ Gemini client initialized")
print(f"✓ Using model: {MODEL}")

%config InlineBackend.figure_format = 'retina'

# Load and Visualize Temporal Image Series

In [ ]:
# Get all brick kiln images and sort by year
image_folder = "brick-kilns"
image_files = sorted(glob.glob(f"{image_folder}/*.png"))

print(f"Found {len(image_files)} images:")
for f in image_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Extract coordinates and years from filenames
def parse_filename(filename):
    """Extract lat, lon, year from filename like 28.212481_77.401398_2014.png"""
    basename = os.path.basename(filename)
    parts = basename.replace('.png', '').split('_')
    return {
        'lat': float(parts[0]),
        'lon': float(parts[1]),
        'year': int(parts[2]),
        'path': filename
    }

# Parse all images
image_data = [parse_filename(f) for f in image_files]
image_data = sorted(image_data, key=lambda x: x['year'])

# Load images
images = [Image.open(item['path']) for item in image_data]
years = [item['year'] for item in image_data]
location = f"{image_data[0]['lat']}°N, {image_data[0]['lon']}°E"

print(f"\nLocation: {location}")
print(f"Years: {years}")
print(f"Time span: {years[-1] - years[0]} years ({years[0]}-{years[-1]})")

In [ ]:
# Visualize the temporal sequence
n_images = len(images)
fig, axes = plt.subplots(1, n_images, figsize=(4*n_images, 4))

if n_images == 1:
    axes = [axes]

for ax, img, year in zip(axes, images, years):
    ax.imshow(img)
    ax.set_title(f"{year}", fontsize=14, fontweight='bold')
    ax.axis('off')

plt.suptitle(f"Temporal Satellite Imagery: {location}", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Multi-Image Temporal Analysis with Gemini 3 Pro

We now send all images in a **single API call** with a comprehensive prompt asking Gemini to analyze temporal patterns across the entire sequence.

In [ ]:
# Construct multi-image prompt with temporal context
contents = []

# Add each image with its year label
for year, img in zip(years, images):
    contents.append(f"**Year {year}:**")
    contents.append(img)

# Comprehensive temporal analysis prompt
prompt = f"""I have provided {len(images)} satellite images of the same location ({location}) taken across different years: {', '.join(map(str, years))}.

These images show the temporal evolution of brick kilns in this region. Please analyze these images chronologically and provide:

1. **First Appearance**: In which year do brick kilns first appear in this location? Describe what you see.

2. **Kiln Type Identification**: For each year where kilns are visible, identify the type(s) of brick kilns present:
   - Bull's Trench Kiln (BTK): Circular or oval shape, continuous operation
   - Fixed Chimney Bull's Trench Kiln (FCBTK): Similar to BTK but with fixed chimneys
   - Zig-zag Kiln: Rectangular, more efficient than BTK
   - Clamp Kiln: Temporary, rectangular mounds
   - Other types

3. **Count and Configuration**: For each year, estimate:
   - Number of kilns visible
   - Approximate size/capacity indicators
   - Spatial arrangement (clustered, linear, scattered)

4. **Operational Status**: For each year, assess whether kilns appear:
   - Active (smoke, dark emissions visible)
   - Inactive/dormant
   - Under construction or abandoned

5. **Temporal Trends**: Describe overall patterns:
   - Growth or decline in number of kilns
   - Changes in kiln types over time
   - Seasonal patterns (if detectable)
   - Any evidence of modernization or technology changes

6. **Environmental Context**: Note any other visible changes:
   - Land use changes around kilns
   - Vegetation changes
   - Infrastructure development

Return your analysis as a structured JSON object with the following format:

{{
  "first_appearance_year": <year>,
  "yearly_analysis": [
    {{
      "year": <year>,
      "kilns_present": <boolean>,
      "kiln_types": [<list of types identified>],
      "kiln_count": <estimated number or "unknown">,
      "operational_status": <"active"|"inactive"|"mixed"|"unknown">,
      "spatial_arrangement": <description>,
      "observations": <detailed notes>
    }}
  ],
  "temporal_trends": {{
    "overall_pattern": <growth|decline|stable|fluctuating>,
    "type_evolution": <description of changes in kiln types>,
    "modernization_evidence": <boolean and description>
  }},
  "environmental_changes": <description>
}}

Be specific and cite visual evidence from the images. If you're uncertain about something, indicate that clearly.
"""

contents.append(prompt)

print(f"Sending {len(images)} images to Gemini 3 Pro for temporal analysis...")
print(f"Years: {years}")
print("\nThis may take 30-60 seconds...\n")

In [ ]:
# Send request to Gemini
import time

start_time = time.time()

response = client.models.generate_content(
    model=MODEL,
    contents=contents
)

elapsed_time = time.time() - start_time

print(f"✓ Analysis complete in {elapsed_time:.2f} seconds")
print(f"\nProcessing rate: {len(images)/elapsed_time:.2f} images/second")

# Analysis Results

In [ ]:
# Display raw response
print("="*80)
print("RAW RESPONSE FROM GEMINI 3 PRO")
print("="*80)
print(response.text)
print("="*80)

In [ ]:
# Parse JSON response
def parse_response(response_text):
    """Extract JSON from response, handling markdown code blocks."""
    try:
        # Remove markdown code blocks if present
        text = response_text.strip()
        if text.startswith('```'):
            # Extract content between ```json and ```
            text = text.split('```', 2)[1]
            if text.startswith('json'):
                text = text[4:].strip()
            text = text.rsplit('```', 1)[0].strip()
        
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Warning: Could not parse JSON response: {e}")
        print(f"Attempting to extract key information manually...")
        return None

analysis_data = parse_response(response.text)

if analysis_data:
    print("✓ Successfully parsed structured analysis")
else:
    print("⚠ Using raw text response (JSON parsing failed)")

## Key Findings

In [ ]:
if analysis_data:
    print("\n" + "="*80)
    print("KEY FINDINGS")
    print("="*80)
    
    print(f"\n📍 Location: {location}")
    print(f"📅 Time Period: {years[0]}-{years[-1]} ({years[-1]-years[0]} years)")
    
    if 'first_appearance_year' in analysis_data:
        print(f"\n🏭 First Appearance: {analysis_data['first_appearance_year']}")
    
    if 'temporal_trends' in analysis_data:
        trends = analysis_data['temporal_trends']
        print(f"\n📈 Overall Pattern: {trends.get('overall_pattern', 'N/A')}")
        print(f"🔄 Type Evolution: {trends.get('type_evolution', 'N/A')}")
        print(f"⚙️ Modernization: {trends.get('modernization_evidence', 'N/A')}")
    
    if 'environmental_changes' in analysis_data:
        print(f"\n🌍 Environmental Changes: {analysis_data['environmental_changes']}")
    
    print("\n" + "="*80)

## Year-by-Year Analysis

In [ ]:
if analysis_data and 'yearly_analysis' in analysis_data:
    print("\n" + "="*80)
    print("YEAR-BY-YEAR BREAKDOWN")
    print("="*80)
    
    for year_data in analysis_data['yearly_analysis']:
        year = year_data.get('year', 'Unknown')
        print(f"\n📅 Year {year}")
        print("-" * 40)
        print(f"  Kilns Present: {'✓ Yes' if year_data.get('kilns_present') else '✗ No'}")
        
        if year_data.get('kilns_present'):
            types = year_data.get('kiln_types', [])
            if types:
                print(f"  Types: {', '.join(types)}")
            
            count = year_data.get('kiln_count', 'unknown')
            print(f"  Count: {count}")
            
            status = year_data.get('operational_status', 'unknown')
            print(f"  Status: {status}")
            
            arrangement = year_data.get('spatial_arrangement', 'N/A')
            print(f"  Arrangement: {arrangement}")
        
        obs = year_data.get('observations', '')
        if obs:
            print(f"  Observations: {obs}")

## Visualization: Temporal Timeline

In [ ]:
if analysis_data and 'yearly_analysis' in analysis_data:
    # Extract data for visualization
    vis_years = []
    kiln_counts = []
    kiln_present = []
    
    for year_data in analysis_data['yearly_analysis']:
        vis_years.append(year_data.get('year'))
        kiln_present.append(1 if year_data.get('kilns_present') else 0)
        
        # Try to extract numeric count
        count = year_data.get('kiln_count', 0)
        if isinstance(count, str):
            # Try to extract number from string
            import re
            match = re.search(r'(\d+)', str(count))
            count = int(match.group(1)) if match else 0
        kiln_counts.append(count if count else 0)
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Plot 1: Kiln presence over time
    ax1.fill_between(vis_years, kiln_present, alpha=0.3, color='orange', label='Kilns Present')
    ax1.plot(vis_years, kiln_present, marker='o', markersize=8, color='darkorange', linewidth=2)
    ax1.set_ylabel('Kiln Presence', fontsize=12)
    ax1.set_ylim(-0.1, 1.1)
    ax1.set_yticks([0, 1])
    ax1.set_yticklabels(['No', 'Yes'])
    ax1.grid(True, alpha=0.3)
    ax1.set_title('Brick Kiln Presence Over Time', fontsize=14, fontweight='bold')
    ax1.legend()
    
    # Plot 2: Kiln count over time
    ax2.bar(vis_years, kiln_counts, color='steelblue', alpha=0.7, edgecolor='darkblue', linewidth=1.5)
    ax2.set_xlabel('Year', fontsize=12)
    ax2.set_ylabel('Number of Kilns', fontsize=12)
    ax2.set_title('Estimated Brick Kiln Count Over Time', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, (year, count) in enumerate(zip(vis_years, kiln_counts)):
        if count > 0:
            ax2.text(year, count + 0.1, str(count), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Visualization skipped: Structured data not available")

# Conclusions

## Key Insights

This analysis demonstrates several important capabilities:

1. **Multi-Image Temporal Context**: Gemini 3 Pro can effectively process multiple satellite images in a single API call, maintaining temporal coherence across years.

2. **Detailed Visual Analysis**: The model can identify specific infrastructure types (kiln types), operational status, and spatial patterns from satellite imagery.

3. **Structured Output**: By requesting JSON format, we can programmatically extract and visualize temporal trends.

4. **Efficiency**: Processing multiple years in a single context is more efficient than sequential analysis and ensures consistent interpretation across the time series.

## Environmental Monitoring Applications

This approach can be applied to:
- **Air Quality Management**: Track pollution sources over time
- **Regulatory Compliance**: Monitor unauthorized constructions
- **Urban Planning**: Assess industrial expansion patterns
- **Climate Studies**: Correlate infrastructure changes with environmental indicators

## Limitations

- Satellite resolution affects detection accuracy
- Seasonal variations (e.g., crop burning, monsoon) can affect visibility
- Ground truth validation recommended for policy decisions
- Model may have uncertainty in distinguishing similar structures

## Future Work

- Test on larger temporal datasets (more locations, longer time spans)
- Compare with traditional computer vision approaches
- Validate findings with ground truth data
- Automate large-scale monitoring across regions

# References

- [Gemini API Documentation](https://ai.google.dev/gemini-api/docs)
- [Previous Post: Batch vs Sequential Processing](https://nipunbatra.github.io/blog/posts/2025-12-12-gemini-batch-vs-sequential.html)
- [Brick Kiln Environmental Impact Studies](https://www.cseindia.org/)
- Location: 28.212481°N, 77.401398°E (Delhi NCR region)